# Subtitle Generator and Summarizer

This notebook covers:
1. Audio extraction from video lectures
2. Transcription using OpenAI Whisper
3. Subtitle (.srt) generation with timestamps
4. Summarization using BART / FLAN-T5
5. Evaluation using WER and ROUGE scores

## Step 0 — Install Dependencies

In [3]:
#install all required packages
!pip install openai-whisper transformers torch torchaudio moviepy rouge-score jiwer accelerate sentencepiece

## Step 1 — Imports and Configuration

In [5]:
import os
import re
import json
import time
import textwrap
import whisper
import torch
from pathlib import Path
from moviepy import VideoFileClip
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer
from jiwer import wer

# ── Configuration ──────────────────────────────────────────────────────────────
CONFIG = {
    "whisper_model": "medium",       # 'base' for speed, 'medium' for accuracy
    "summarizer_model": "facebook/bart-large-cnn",  # or 'google/flan-t5-base'
    "max_summary_words": 100,
    "chunk_duration_sec": 120,       # split transcript every ~2 minutes
    "output_dir": "outputs",
    "video_dir": "samples",          # put your .mp4/.mkv/.avi files here
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print("Configuration loaded ✓")

Using device: cpu
Configuration loaded ✓


## Step 2 — Audio Extraction

In [9]:
import os
os.chdir(r"C:\Users\Neha Vishwakarma\OneDrive\Desktop\subtitle_summarizer")

# Create directories if they don't exist
os.makedirs(CONFIG["video_dir"], exist_ok=True)
os.makedirs(CONFIG["output_dir"], exist_ok=True)

def extract_audio(video_path: str, output_dir: str) -> str:
    """
    Extract audio from a video file and save as .wav.
    Returns path to the extracted audio file.
    """
    video_path = Path(video_path)
    audio_path = Path(output_dir) / (video_path.stem + ".wav")

    if audio_path.exists():
        print(f"  [skip] Audio already exists: {audio_path.name}")
        return str(audio_path)

    print(f"  Extracting audio from: {video_path.name}")
    clip = VideoFileClip(str(video_path))
    clip.audio.write_audiofile(str(audio_path), verbose=False, logger=None)
    clip.close()

    print(f"  Audio saved to: {audio_path}")
    return str(audio_path)

# Gather all video files in the samples directory
video_files = sorted([
    str(p) for p in Path(CONFIG["video_dir"]).iterdir()
    if p.suffix.lower() in (".mp4", ".mkv", ".avi", ".mov", ".webm")
])

print(f"Found {len(video_files)} video(s):")
for v in video_files:
    print(f"  • {v}")

# Extract audio for all videos
audio_files = {}
for vf in video_files:
    name = Path(vf).stem
    audio_files[name] = extract_audio(vf, CONFIG["output_dir"])

print("\nAudio extraction complete ✓")

Found 3 video(s):
  • samples\lecture1.mp4
  • samples\lecture2.mp4
  • samples\lecture3.mp4
  [skip] Audio already exists: lecture1.wav
  [skip] Audio already exists: lecture2.wav
  [skip] Audio already exists: lecture3.wav

Audio extraction complete ✓


## Step 3 — Transcription with Whisper

In [17]:
import time

# ── Reload existing transcripts first ─────────────────────────────────────────
transcripts = {}

for txt_file in Path(CONFIG["output_dir"]).glob("*_transcript.txt"):
    name = txt_file.stem.replace("_transcript", "")
    transcripts[name] = {
        "text": txt_file.read_text(encoding="utf-8"),
        "segments": []
    }
    print(f"  [loaded] {txt_file.name}")

already_done = set(transcripts.keys())
remaining = [vf for vf in audio_files if vf not in already_done]

if not remaining:
    print(f"\nAll {len(transcripts)} transcript(s) loaded from disk ✓")
    print("Skipping Whisper — nothing new to transcribe.")

else:
    print(f"\n{len(already_done)} loaded from disk, {len(remaining)} need transcription.")
    print(f"Loading Whisper '{CONFIG['whisper_model']}' model...")
    whisper_model = whisper.load_model(CONFIG["whisper_model"])
    print("Whisper loaded ✓\n")

    def transcribe_audio(audio_path: str) -> dict:
        print(f"  Transcribing: {Path(audio_path).name}")
        start = time.time()
        result = whisper_model.transcribe(
            audio_path,
            language="en",
            verbose=False,
            word_timestamps=False,
        )
        elapsed = time.time() - start
        n_segs = len(result["segments"])
        print(f"  → {n_segs} segments | {len(result['text'].split())} words | {elapsed/60:.1f} min")
        return result

    for name in remaining:
        audio_path = audio_files[name]
        transcripts[name] = transcribe_audio(audio_path)
        txt_path = Path(CONFIG["output_dir"]) / f"{name}_transcript.txt"
        txt_path.write_text(transcripts[name]["text"], encoding="utf-8")
        print(f"  Transcript saved: {txt_path}\n")

print(f"\nTranscription complete ✓  ({len(transcripts)} total)")

  [loaded] lecture1_transcript.txt
  [loaded] lecture2_transcript.txt
  [loaded] lecture3_transcript.txt

All 3 transcript(s) loaded from disk ✓
Skipping Whisper — nothing new to transcribe.

Transcription complete ✓  (3 total)


In [18]:
# Reload existing outputs instead of re-transcribing
transcripts = {}
summaries = {}

for txt_file in Path(CONFIG["output_dir"]).glob("*_transcript.txt"):
    name = txt_file.stem.replace("_transcript", "")
    transcripts[name] = {"text": txt_file.read_text(encoding="utf-8"), "segments": []}
    print(f"  Loaded transcript: {txt_file.name}")

for sum_file in Path(CONFIG["output_dir"]).glob("*_summary.txt"):
    name = sum_file.stem.replace("_summary", "")
    summaries[name] = sum_file.read_text(encoding="utf-8")
    print(f"  Loaded summary:    {sum_file.name}")

print(f"\nLoaded {len(transcripts)} transcript(s), {len(summaries)} summary/summaries ✓")

  Loaded transcript: lecture1_transcript.txt
  Loaded transcript: lecture2_transcript.txt
  Loaded transcript: lecture3_transcript.txt
  Loaded summary:    lecture1_summary.txt
  Loaded summary:    lecture2_summary.txt
  Loaded summary:    lecture3_summary.txt
  Loaded summary:    sample_lecture_demo_summary.txt

Loaded 3 transcript(s), 4 summary/summaries ✓


## Step 4 — Generate .srt Subtitle Files

In [25]:
def seconds_to_srt_time(seconds: float) -> str:
    ms = int(round((seconds % 1) * 1000))
    s = int(seconds)
    h = s // 3600
    m = (s % 3600) // 60
    s = s % 60
    return f"{h:02}:{m:02}:{s:02},{ms:03}"


def text_to_srt(text: str, words_per_sub: int = 12, wpm: int = 130) -> str:
    """Generate SRT from plain text using estimated timestamps."""
    words = text.split()
    if not words:
        return ""
    secs_per_word = 60 / wpm
    blocks = []
    idx = 1
    pos = 0
    while pos < len(words):
        chunk = words[pos: pos + words_per_sub]
        start_sec = pos * secs_per_word
        end_sec = (pos + len(chunk)) * secs_per_word
        line = " ".join(chunk)
        blocks.append(f"{idx}\n{seconds_to_srt_time(start_sec)} --> {seconds_to_srt_time(end_sec)}\n{line}\n")
        idx += 1
        pos += words_per_sub
    return "\n".join(blocks)


# Generate SRT for all videos
for name, result in transcripts.items():
    text = result["text"].strip()
    if result["segments"]:
        # Use real Whisper segments if available
        from textwrap import fill
        srt_blocks = []
        for i, seg in enumerate(result["segments"], 1):
            start = seconds_to_srt_time(seg["start"])
            end = seconds_to_srt_time(seg["end"])
            srt_blocks.append(f"{i}\n{start} --> {end}\n{fill(seg['text'].strip(), 80)}\n")
        srt_content = "\n".join(srt_blocks)
    else:
        # Fall back to estimated timestamps from plain text
        srt_content = text_to_srt(text)

    srt_path = Path(CONFIG["output_dir"]) / f"{name}.srt"
    srt_path.write_text(srt_content, encoding="utf-8")
    n_entries = srt_content.count("\n\n")
    print(f"SRT saved: {srt_path}  ({n_entries} entries)")

print("\nSubtitle generation complete ✓")

SRT saved: outputs\lecture1.srt  (234 entries)
SRT saved: outputs\lecture2.srt  (334 entries)
SRT saved: outputs\lecture3.srt  (3025 entries)

Subtitle generation complete ✓


## Step 5 — Chunk Transcript into Logical Sections

In [20]:
def chunk_text_by_words(text: str, chunk_duration: float = 120, wpm: int = 130) -> list:
    """Chunk plain text into time-based segments using estimated timestamps."""
    words = text.split()
    if not words:
        return []
    words_per_chunk = int(wpm * (chunk_duration / 60))
    chunks = []
    for i in range(0, len(words), words_per_chunk):
        chunk_words = words[i: i + words_per_chunk]
        start_sec = i * (60 / wpm)
        end_sec = (i + len(chunk_words)) * (60 / wpm)
        chunks.append({
            "text": " ".join(chunk_words),
            "start": start_sec,
            "end": end_sec,
        })
    return chunks


chunked = {}
for name, result in transcripts.items():
    if result["segments"]:
        # Use real segments if available
        chunks = chunk_segments_by_time(result["segments"], CONFIG["chunk_duration_sec"])
    else:
        # Fall back to word-based chunking from plain text
        chunks = chunk_text_by_words(result["text"], CONFIG["chunk_duration_sec"])
    chunked[name] = chunks
    print(f"{name}: {len(chunks)} chunk(s)")

print("\nChunking complete ✓")

lecture1: 0 chunk(s)
lecture2: 0 chunk(s)
lecture3: 0 chunk(s)

Chunking complete ✓


In [26]:
# Confirm what we have loaded
print("Transcripts loaded:", list(transcripts.keys()))
print("Summaries loaded:", list(summaries.keys()))
print()

# Re-confirm summaries are populated
for name, summary in summaries.items():
    if "_demo" in name:
        continue
    word_count = len(summary.split())
    print(f"{name}: {word_count} words")
    print(f"  Preview: {summary[:100]}...")
    print()# Confirm what we have loaded
print("Transcripts loaded:", list(transcripts.keys()))
print("Summaries loaded:", list(summaries.keys()))
print()

# Re-confirm summaries are populated
for name, summary in summaries.items():
    if "_demo" in name:
        continue
    word_count = len(summary.split())
    print(f"{name}: {word_count} words")
    print(f"  Preview: {summary[:100]}...")
    print()

Transcripts loaded: ['lecture1', 'lecture2', 'lecture3']
Summaries loaded: ['lecture1', 'lecture2', 'lecture3']

lecture1: 0 words
  Preview: ...

lecture2: 0 words
  Preview: ...

lecture3: 0 words
  Preview: ...

Transcripts loaded: ['lecture1', 'lecture2', 'lecture3']
Summaries loaded: ['lecture1', 'lecture2', 'lecture3']

lecture1: 0 words
  Preview: ...

lecture2: 0 words
  Preview: ...

lecture3: 0 words
  Preview: ...



## Step 6 — Summarization using BART / FLAN-T5

In [23]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

print("Loading summarizer: facebook/bart-large-cnn")

sum_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
sum_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")
sum_model.eval()
print("Summarizer loaded ✓\n")


def summarizer(text, max_length=80, min_length=20, **kwargs):
    inputs = sum_tokenizer(
        text,
        return_tensors="pt",
        max_length=1024,
        truncation=True,
    )
    with torch.no_grad():
        output = sum_model.generate(
            **inputs,
            max_length=max_length,
            min_length=min_length,
            num_beams=4,
            early_stopping=True,
        )
    summary = sum_tokenizer.decode(output[0], skip_special_tokens=True)
    return [{"summary_text": summary}]


def summarize_chunks(chunks: list, max_words: int = 100) -> str:
    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        text = chunk["text"].strip()
        if len(text.split()) < 30:
            chunk_summaries.append(text)
            continue
        input_len = len(text.split())
        max_out = min(80, max(30, input_len // 3))
        min_out = min(20, max_out - 10)
        result = summarizer(text, max_length=max_out, min_length=min_out)
        chunk_summaries.append(result[0]["summary_text"])
        print(f"    Chunk {i+1}/{len(chunks)} summarized.")

    combined = " ".join(chunk_summaries)
    words = combined.split()
    if len(words) > max_words:
        combined = " ".join(words[:max_words]) + "..."
    return combined


# Generate summaries for all videos
summaries = {}
for name, chunks in chunked.items():
    print(f"Summarizing: {name}")
    summary = summarize_chunks(chunks, CONFIG["max_summary_words"])
    summaries[name] = summary
    summary_path = Path(CONFIG["output_dir"]) / f"{name}_summary.txt"
    summary_path.write_text(summary, encoding="utf-8")
    print(f"  Summary saved: {summary_path}")
    print(f"  Preview: {summary[:120]}...\n")

print("Summarization complete ✓")

Loading summarizer: facebook/bart-large-cnn


vocab.json: 0.00B [00:00, ?B/s]

C:\Users\Neha Vishwakarma\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Neha Vishwakarma\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Summarizer loaded ✓

Summarizing: lecture1
  Summary saved: outputs\lecture1_summary.txt
  Preview: ...

Summarizing: lecture2
  Summary saved: outputs\lecture2_summary.txt
  Preview: ...

Summarizing: lecture3
  Summary saved: outputs\lecture3_summary.txt
  Preview: ...

Summarization complete ✓


## Step 7 — Evaluation: WER and ROUGE Scores

In [24]:
#  WER Evaluation 
# WER compares Whisper's transcription against a ground-truth reference.
# To use: provide reference transcripts in the dict below.
# If you have no reference, WER section will be skipped gracefully.

reference_transcripts = {
    # "video_name": "Ground truth transcript text here...",
    # Add entries matching your video stem names.
}

wer_scores = {}
if reference_transcripts:
    print("── Word Error Rate (WER) ──")
    for name, ref_text in reference_transcripts.items():
        if name in transcripts:
            hyp_text = transcripts[name]["text"]
            score = wer(ref_text, hyp_text)
            wer_scores[name] = round(score, 4)
            print(f"  {name}: WER = {score:.2%}")
else:
    print("No reference transcripts provided — WER skipped.")
    print("Add ground-truth text to `reference_transcripts` dict above to compute WER.")


# ── ROUGE Evaluation ──────────────────────────────────────────────────────────
# ROUGE compares model summaries against reference summaries.
reference_summaries = {
    # "video_name": "Reference summary here...",
}

rouge_scores = {}
scorer_obj = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

if reference_summaries:
    print("\n── ROUGE Scores ──")
    for name, ref_sum in reference_summaries.items():
        if name in summaries:
            scores = scorer_obj.score(ref_sum, summaries[name])
            rouge_scores[name] = {
                "rouge1_f": round(scores["rouge1"].fmeasure, 4),
                "rouge2_f": round(scores["rouge2"].fmeasure, 4),
                "rougeL_f": round(scores["rougeL"].fmeasure, 4),
            }
            print(f"  {name}: ROUGE-1={scores['rouge1'].fmeasure:.3f}, "
                  f"ROUGE-2={scores['rouge2'].fmeasure:.3f}, "
                  f"ROUGE-L={scores['rougeL'].fmeasure:.3f}")
else:
    print("\nNo reference summaries provided — ROUGE skipped.")
    print("Add reference text to `reference_summaries` dict above to compute ROUGE.")


# ── Save Evaluation Report ────────────────────────────────────────────────────
report = {
    "wer_scores": wer_scores,
    "rouge_scores": rouge_scores,
    "summaries": summaries,
    "model_config": CONFIG,
}
report_path = Path(CONFIG["output_dir"]) / "evaluation_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"\nEvaluation report saved: {report_path} ✓")

No reference transcripts provided — WER skipped.
Add ground-truth text to `reference_transcripts` dict above to compute WER.

No reference summaries provided — ROUGE skipped.
Add reference text to `reference_summaries` dict above to compute ROUGE.

Evaluation report saved: outputs\evaluation_report.json ✓


## Step 8 — Display Final Summary

In [22]:
print("═" * 60)
print("FINAL OUTPUT SUMMARY")
print("═" * 60)

for name in transcripts:
    print(f"\n📹  {name}")
    print(f"  Transcript : outputs/{name}_transcript.txt")
    print(f"  Subtitle   : outputs/{name}.srt")
    print(f"  Summary    : outputs/{name}_summary.txt")
    print(f"  ─── Summary preview ───")
    print(textwrap.fill(summaries.get(name, "(none)"), width=70, initial_indent="  ", subsequent_indent="  "))
    word_count = len(summaries.get(name, "").split())
    print(f"  Word count: {word_count}")

print("\n" + "═" * 60)
print("All files saved to ./outputs/")
print("Run zip_outputs.py to bundle everything for submission.")
print("═" * 60)

════════════════════════════════════════════════════════════
FINAL OUTPUT SUMMARY
════════════════════════════════════════════════════════════

📹  lecture1
  Transcript : outputs/lecture1_transcript.txt
  Subtitle   : outputs/lecture1.srt
  Summary    : outputs/lecture1_summary.txt
  ─── Summary preview ───
  Claude code is an AI coding tool. Claude code is an AI coding tool.
  Claude code is an AI coding tool. Learn to use Claude code. Learn to
  use Claude code. Learn to use Claude code. Learn to use Claude code.
  Create a website. Test the website. Learn the coding. You can go to
  an existing expense. You can use the AI codex to learn how to code.
  Claude code is the most powerful AI coding tool. You will be able to
  create your own opinions and make your own opinions. If you are a
  software developer or a student,...
  Word count: 100

📹  lecture2
  Transcript : outputs/lecture2_transcript.txt
  Subtitle   : outputs/lecture2.srt
  Summary    : outputs/lecture2_summary.txt
  ──

In [27]:
import os

output_dir = CONFIG["output_dir"]
print("Files in outputs/:\n")
for f in sorted(os.listdir(output_dir)):
    size = os.path.getsize(os.path.join(output_dir, f))
    print(f"  {f:45s}  {size:>8} bytes")

Files in outputs/:

  evaluation_report.json                              365 bytes
  lecture1.srt                                      23673 bytes
  lecture1.wav                                   178843218 bytes
  lecture1_summary.txt                                  0 bytes
  lecture1_transcript.txt                           14618 bytes
  lecture2.srt                                      33602 bytes
  lecture2.wav                                   281469210 bytes
  lecture2_summary.txt                                  0 bytes
  lecture2_transcript.txt                           20647 bytes
  lecture3.srt                                     320548 bytes
  lecture3.wav                                   2057469702 bytes
  lecture3_summary.txt                                  0 bytes
  lecture3_transcript.txt                          200617 bytes
  sample_lecture_demo.srt                            1393 bytes
  sample_lecture_demo_summary.txt                     544 bytes


In [28]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

print("Loading BART...")
sum_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
sum_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")
sum_model.eval()
print("BART loaded ✓\n")

def summarize_text(text: str, max_words: int = 100) -> str:
    # Split text into chunks of ~400 words each
    words = text.split()
    chunk_size = 400
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]
    
    parts = []
    for i, chunk in enumerate(chunks):
        if len(chunk.split()) < 30:
            parts.append(chunk)
            continue
        inputs = sum_tokenizer(chunk, return_tensors="pt", max_length=1024, truncation=True)
        with torch.no_grad():
            output = sum_model.generate(
                **inputs,
                max_length=80,
                min_length=20,
                num_beams=4,
                early_stopping=True,
            )
        summary = sum_tokenizer.decode(output[0], skip_special_tokens=True)
        parts.append(summary)
        print(f"  chunk {i+1}/{len(chunks)} done")

    combined = " ".join(parts)
    words = combined.split()
    if len(words) > max_words:
        combined = " ".join(words[:max_words]) + "..."
    return combined

# Summarize all 3 videos from transcript files
for name in ["lecture1", "lecture2", "lecture3"]:
    txt_path = Path(CONFIG["output_dir"]) / f"{name}_transcript.txt"
    text = txt_path.read_text(encoding="utf-8").strip()
    print(f"\nSummarizing {name} ({len(text.split())} words)...")


    
    summary = summarize_text(text, max_words=100)
    summaries[name] = summary
    
    sum_path = Path(CONFIG["output_dir"]) / f"{name}_summary.txt"
    sum_path.write_text(summary, encoding="utf-8")
    print(f"  Saved: {sum_path}")
    print(f"  Preview: {summary[:120]}")

print("\nSummarization complete ✓")

Loading BART...


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

BART loaded ✓


Summarizing lecture1 (2817 words)...
  chunk 1/8 done
  chunk 2/8 done
  chunk 3/8 done
  chunk 4/8 done
  chunk 5/8 done
  chunk 6/8 done
  chunk 7/8 done
  Saved: outputs\lecture1_summary.txt
  Preview: Nitish: I'm playing a new playlist on this channel on a Claude code. Claude is an AI coding tool from Anthropic. Nitish:

Summarizing lecture2 (4016 words)...
  chunk 1/11 done
  chunk 2/11 done
  chunk 3/11 done
  chunk 4/11 done
  chunk 5/11 done
  chunk 6/11 done
  chunk 7/11 done
  chunk 8/11 done
  chunk 9/11 done
  chunk 10/11 done
  Saved: outputs\lecture2_summary.txt
  Preview: In this video, Nitish shows you how to install Claude code on your machine. To use Claude code, you have to pay the end-

Summarizing lecture3 (36305 words)...
  chunk 1/91 done
  chunk 2/91 done
  chunk 3/91 done
  chunk 4/91 done
  chunk 5/91 done
  chunk 6/91 done
  chunk 7/91 done
  chunk 8/91 done
  chunk 9/91 done
  chunk 10/91 done
  chunk 11/91 done
  chunk 12/91 done
  chunk 13/